In [1]:
from resemblyzer import VoiceEncoder, preprocess_wav
import numpy as np



In [11]:
def speaker_similarity(file1, file2):
    encoder = VoiceEncoder()
    
    wav1 = preprocess_wav(file1)
    wav2 = preprocess_wav(file2)
    
    emb1 = encoder.embed_utterance(wav1)
    emb2 = encoder.embed_utterance(wav2)
    
    emb1 /= np.linalg.norm(emb1)
    emb2 /= np.linalg.norm(emb2)
    
    return np.dot(emb1, emb2)

score = speaker_similarity("part_031_s2.wav", "part_002_s2.wav")
print(score)


Loaded the voice encoder model on cpu in 0.03 seconds.
0.9051236


In [4]:
import numpy as np
from resemblyzer import VoiceEncoder, preprocess_wav
from pathlib import Path

def implement_centroid_detection(reference_files, test_file, threshold=0.85):
    encoder = VoiceEncoder()
    
    # --- Step 1: Create the Reference Centroid ---
    ref_embeddings = []
    for f_path in reference_files:
        wav = preprocess_wav(f_path)
        emb = encoder.embed_utterance(wav)
        # Normalize the embedding (optional, but good practice for cosine similarity)
        emb /= np.linalg.norm(emb)
        ref_embeddings.append(emb)
    
    # Calculate the mathematical average (The Centroid)
    centroid = np.mean(ref_embeddings, axis=0)
    # Re-normalize the centroid to ensure it's still a unit vector
    centroid /= np.linalg.norm(centroid)
    
    # --- Step 2: Process the Test Audio ---
    test_wav = preprocess_wav(test_file)
    test_emb = encoder.embed_utterance(test_wav)
    test_emb /= np.linalg.norm(test_emb)
    
    # --- Step 3: Calculate Similarity ---
    # Dot product between normalized vectors is the Cosine Similarity
    score = np.dot(test_emb, centroid)
    
    # --- Step 4: Decision ---
    is_real = score >= threshold
    
    return score, is_real

# Example Usage:
# list of many real recordings of Speaker A
reference_recordings =[]
for i in range(29):
    if i < 10:
        name  = "speaker1/speaker1/part_00"+str(i)+".wav"
    else:
        name  = "speaker1/speaker1/part_0"+str(i)+".wav"
    reference_recordings.append(name)

suspicious_clip = "deepfake_audio.wav"

score, result = implement_centroid_detection(reference_recordings, suspicious_clip)
print(f"Similarity Score: {score:.4f}")
print(f"Verdict: {'REAL' if result else 'DEEPFAKE'}")

Loaded the voice encoder model on cpu in 0.01 seconds.
Similarity Score: 0.7476
Verdict: DEEPFAKE
